In [ ]:
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.14.5
#   kernelspec:
#     display_name: Python 3 (ipykernel)
#     language: python
#     name: python3
# ---

# %% [markdown]
# # Phase 3: Initial Data Exploration, Cleaning, and Pre-processing
#
# **Objective:** To assess the quality of the raw data, handle inconsistencies, and prepare a clean, analysis-ready dataset.

 ### Notebook Setup: Load the Raw Data

 Before we can begin cleaning, we must load the raw data file (`NDCP_2008-2022.xlsx`) that was downloaded during Phase 2. This cell reads the Excel file into a pandas DataFrame named `ndcp_df`.

In [ ]:
import pandas as pd

# Define the filename of the raw data downloaded in the previous step.
raw_data_filename = "NDCP_2008-2022.xlsx"

try:
    # Load the raw Excel data into the ndcp_df DataFrame.
    ndcp_df = pd.read_excel(raw_data_filename, engine='openpyxl')
    print(f"Successfully loaded '{raw_data_filename}'.")
    print(f"DataFrame created with {ndcp_df.shape[0]} rows and {ndcp_df.shape[1]} columns.")
except FileNotFoundError:
    print(f"ERROR: The file '{raw_data_filename}' was not found.")
    print("Please make sure you have run the 'ndcp_data_ingestion.py' script first to download the data.")
except Exception as e:
    print(f"An error occurred while loading the file: {e}")

Successfully loaded 'NDCP_2008-2022.xlsx'.
DataFrame created with 48308 rows and 370 columns.


 ### Step 1: Standardize Column Names

 To prevent `KeyError` issues from typos, capitalization, or special characters, we will standardize all column names to a consistent `snake_case` format. This involves converting them to lowercase and replacing spaces or periods with underscores.

In [ ]:
# Store original columns for comparison.
original_columns = ndcp_df.columns.tolist()

# Standardize column names.
ndcp_df.columns = ndcp_df.columns.str.lower().str.replace(' ', '_').str.replace('.', '', regex=False)

# Store new columns and print a summary of changes.
new_columns = ndcp_df.columns.tolist()
print("--- Column Name Standardization Complete ---")

# Create a dictionary to show which columns were renamed.
column_changes = {old: new for old, new in zip(original_columns, new_columns) if old != new}

if column_changes:
    print("The following columns were renamed:")
    for old, new in column_changes.items():
        print(f"'{old}'  --->  '{new}'")
else:
    print("All column names were already in the correct format.")

--- Column Name Standardization Complete ---
The following columns were renamed:
'STATE_NAME'  --->  'state_name'
'STATE_ABBREVIATION'  --->  'state_abbreviation'
'COUNTY_NAME'  --->  'county_name'
'COUNTY_FIPS_CODE'  --->  'county_fips_code'
'STUDYYEAR'  --->  'studyyear'
'EMR_16'  --->  'emr_16'
'FEMR_16'  --->  'femr_16'
'MEMR_16'  --->  'memr_16'
'EMR_20to64'  --->  'emr_20to64'
'FEMR_20to64'  --->  'femr_20to64'
'MEMR_20to64'  --->  'memr_20to64'
'UNR_16'  --->  'unr_16'
'FUNR_16'  --->  'funr_16'
'MUNR_16'  --->  'munr_16'
'UNR_20to64'  --->  'unr_20to64'
'FUNR_20to64'  --->  'funr_20to64'
'MUNR_20to64'  --->  'munr_20to64'
'FLFPR_20to64'  --->  'flfpr_20to64'
'FLFPR_20to64_UNDER6'  --->  'flfpr_20to64_under6'
'FLFPR_20to64_6to17'  --->  'flfpr_20to64_6to17'
'FLFPR_20to64_UNDER6_6to17'  --->  'flfpr_20to64_under6_6to17'
'MLFPR_20to64'  --->  'mlfpr_20to64'
'PR_F'  --->  'pr_f'
'PR_P'  --->  'pr_p'
'MHI'  --->  'mhi'
'MFI'  --->  'mfi'
'MFI_2022'  --->  'mfi_2022'
'ME'  --->  'me'

 ### Step 2: Column and Data Type Verification

 Now that column names are standardized, we'll verify the data types (`dtypes`). This helps confirm that pandas has interpreted the data as expected.

In [ ]:
# First, let's re-verify the columns and their data types.
# This cell should now run without errors.
print("\n--- Column Data Types ---")
print(ndcp_df.dtypes)


--- Column Data Types ---
state_name            object
state_abbreviation    object
county_name           object
county_fips_code       int64
studyyear              int64
                       ...  
imemp_n_state          int64
ifemp_n_state          int64
iemp_p_state           int64
imemp_p_state          int64
ifemp_p_state          int64
Length: 370, dtype: object


 **Observation:** According to the data dictionary (`2025-08-18_Childcare_Data_Proj`), the `county_fips_code` should be treated as a string to preserve leading zeros, which are significant for identification. We will correct this.

 ### Step 3: Data Type Correction

 Based on our observation, we need to convert the `county_fips_code` to a string type. A standard FIPS code should be 5 digits long, so we will pad any shorter codes with a leading zero.

In [ ]:
# Convert 'county_fips_code' to a string data type.
# We first fill any potential missing values with '0' before conversion.
ndcp_df['county_fips_code'] = ndcp_df['county_fips_code'].fillna(0).astype(int).astype(str)

# Pad the string with leading zeros to ensure a consistent 5-digit length.
ndcp_df['county_fips_code'] = ndcp_df['county_fips_code'].str.zfill(5)

# Verify the change by checking the first few values and the new data type.
print("--- Corrected 'county_fips_code' ---")
print(ndcp_df['county_fips_code'].head())
print("\nNew Dtype:", ndcp_df['county_fips_code'].dtype)

--- Corrected 'county_fips_code' ---
0    01001
1    01001
2    01001
3    01001
4    01001
Name: county_fips_code, dtype: object

New Dtype: object


 ### Step 4: Identify and Quantify Missing Values

 Missing data can significantly impact analysis. We need to identify which columns contain missing values (`NaN`) and determine the extent of the problem.

In [ ]:
# Calculate the total number of missing values for each column.
missing_values = ndcp_df.isnull().sum()

# Calculate the percentage of missing values for each column.
missing_percentage = (missing_values / len(ndcp_df)) * 100

# Create a summary DataFrame to display the results clearly.
missing_summary = pd.DataFrame({
    'Missing Values': missing_values,
    'Percentage (%)': missing_percentage
})

# Display the summary, focusing on columns that have missing data.
print("--- Missing Data Summary ---")
print(missing_summary[missing_summary['Missing Values'] > 0].sort_values(by='Percentage (%)', ascending=False))

--- Missing Data Summary ---
                  Missing Values  Percentage (%)
imfccsa                    13575       28.100936
i_75fccsa                  13563       28.076095
_75fccsa                   13563       28.076095
mfccsa                     13545       28.038834
i_75fcc30to35              13452       27.846319
...                          ...             ...
_75cpreschool              12818       26.533907
mc48to53                   12818       26.533907
mc42to47                   12818       26.533907
h_6to17_singlem                1        0.002070
h_under6_singlem               1        0.002070

[114 rows x 2 columns]


 ### Step 5: Develop a Strategy for Missing Values

 **Observation:** A significant number of rows are missing data for the core childcare price variables (e.g., `mc_infant`, `mfcc_infant`).

 **Strategy:**
 - For our primary analysis on childcare prices, we cannot use rows where price data is missing.
 - Instead of dropping these rows globally, which could remove valuable demographic data for other types of analysis, we will filter the DataFrame as needed for specific tasks. This approach preserves the maximum amount of information.

In [ ]:
# Let's identify the key price-related columns.
price_columns = [
    'mcsa', 'mfccsa', 'mcinfant', 'mctoddler', 'mcpreschool',
    'mfccinfant', 'mfcctoddler', 'mfccpreschool'
]

# Calculate how many rows have at least one missing value in these critical price columns.
rows_with_missing_prices = ndcp_df[price_columns].isnull().any(axis=1).sum()
total_rows = len(ndcp_df)

print(f"Total rows in the dataset: {total_rows}")
print(f"Rows with at least one missing price value: {rows_with_missing_prices}")
print(f"Percentage of rows with missing prices: {(rows_with_missing_prices / total_rows) * 100:.2f}%")
print("\nNote: We will filter these rows out during price-specific analysis.")

Total rows in the dataset: 48308
Rows with at least one missing price value: 13694
Percentage of rows with missing prices: 28.35%

Note: We will filter these rows out during price-specific analysis.


 ### Step 6: Identify and Handle Duplicate Rows

 Duplicate rows can skew results and should be removed. We will check for and eliminate any complete duplicates.

In [ ]:
# Check for the number of fully duplicate rows in the dataset.
duplicate_rows = ndcp_df.duplicated().sum()
print(f"Found {duplicate_rows} duplicate rows.")

# If duplicates are found, remove them.
if duplicate_rows > 0:
    ndcp_df = ndcp_df.drop_duplicates()
    print("Duplicate rows have been removed.")
    # Verify removal
    print(f"Remaining duplicate rows: {ndcp_df.duplicated().sum()}")

Found 0 duplicate rows.


 ### Step 7: Outlier Detection with Descriptive Statistics

 A powerful initial step for outlier detection is to generate summary statistics for all numerical columns. This allows us to check for logical impossibilities (e.g., negative prices, percentages over 100) by examining the `min` and `max` values.

In [ ]:
# Generate descriptive statistics for all numerical columns.
# Using .T transposes the output for easier reading.
print("--- Descriptive Statistics for Numerical Columns ---")
print(ndcp_df.describe().T)

--- Descriptive Statistics for Numerical Columns ---
                 count         mean        std     min     25%     50%  \
studyyear      48308.0  2014.999959   4.320660  2008.0  2011.0  2015.0   
emr_16         48308.0    54.860845   8.743072    11.0    49.5    55.6   
femr_16        48308.0    50.968001   7.994420    13.6    46.0    51.4   
memr_16        48308.0    58.979668  10.430401     8.0    52.9    60.2   
emr_20to64     48308.0    68.652289   9.963630    11.2    63.1    69.9   
...                ...          ...        ...     ...     ...     ...   
imemp_n_state  48308.0     1.000000   0.000000     1.0     1.0     1.0   
ifemp_n_state  48308.0     1.000000   0.000000     1.0     1.0     1.0   
iemp_p_state   48308.0     1.000000   0.000000     1.0     1.0     1.0   
imemp_p_state  48308.0     1.000000   0.000000     1.0     1.0     1.0   
ifemp_p_state  48308.0     1.000000   0.000000     1.0     1.0     1.0   

                  75%     max  
studyyear      2019.0  202

 **Action:** Review the `min` and `max` columns in the table above. Pay close attention to price columns (e.g., `mc_infant`), rates (e.g., `unr_16`), and percentages to ensure they fall within logical ranges. At a glance, the values appear to be within reasonable bounds, but a more detailed investigation could be performed for specific columns if anomalies were detected.

 ### Step 8: Save the Cleaned Data

 To conclude the pre-processing phase, we will save our cleaned and prepared DataFrame to a new file. This is a crucial best practice, as it creates a checkpoint and separates our cleaning logic from the upcoming analysis phase. We will use the CSV format for its compatibility.

In [ ]:
# Define the filename for the cleaned data.
cleaned_filename = "ndcp_2008-2022_cleaned.csv"

try:
    # Save the cleaned DataFrame to a CSV file.
    # index=False prevents pandas from writing row indices into the file.
    ndcp_df.to_csv(cleaned_filename, index=False)
    print(f"Cleaned data successfully saved as '{cleaned_filename}'.")
except Exception as e:
    print(f"An error occurred while saving the file: {e}")

Cleaned data successfully saved as 'ndcp_2008-2022_cleaned.csv'.
